<a href="https://colab.research.google.com/github/Whilmis/Admision_actual/blob/main/Copia_de_whilmisA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Escucha de Objetos y Simulación

La sesión está activa. Si recibes un objeto como `{"request_id":"..."}`, pégalo aquí para que pueda analizar a qué endpoint o consulta de GraphQL corresponde y simular la respuesta adecuada.

### Blackjack Automation: Unified Single-Sheet Script
This script performs the following flow:
1. **Auth:** Logs in and captures the AuthToken.
2. **Setup:** Mimics browser asset loading and WebSocket handshakes.
3. **Loop:** Plays 5 hands with a 1€ bet using Basic Strategy.
4. **Telemetry:** Sends background heartbeats to Datadog to avoid bot detection.
5. **Exit:** Sends finalization signals to ensure the server saves the session state.

### Proceso de Login y Verificación (Celda Separada)

### Login Dinámico (Obtener `AuthTokenV2`)

In [ ]:
import requests
import json
import uuid # Importar el módulo uuid
import random # Importar random para datadog IDs

LEOVEGAS_EMAIL = 'whilmisweb@gmail.com'
LEOVEGAS_PASSWORD = 'Wpdp199659**'

def perform_login(email, password):
    login_url = "https://www.leovegas.es/api/graphql"

    # Generate dynamic trace and request IDs
    req_id = str(uuid.uuid4())
    trace_id = str(random.randint(10**18, 10**19 - 1))
    parent_id = str(random.randint(10**18, 10**19 - 1))
    # Format traceparent according to W3C Trace Context
    traceparent = f"00-{trace_id}{parent_id}-01"

    login_headers = {
        "accept": "*/*",
        "accept-language": "es-US,es-419;q=0.9,es;q=0.8",
        "cache-control": "no-cache",
        "content-type": "application/json",
        "pragma": "no-cache",
        "priority": "u=1, i",
        "sec-ch-ua": "\"Not;A=Brand\";v=\"8\", \"Chromium\";v=\"150\", \"Google Chrome\";v=\"150\"",
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": "\"Windows\"",
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "traceparent": traceparent, # Dynamic traceparent
        "tracestate": "dd=s:1;o:rum",
        "x-datadog-origin": "rum",
        "x-datadog-parent-id": parent_id, # Dynamic parent ID
        "x-datadog-sampling-priority": "1",
        "x-datadog-trace-id": trace_id, # Dynamic trace ID
        "x-domain": "www.leovegas.es",
        "x-language": "es-es",
        "x-locale": "es-es",
        "x-market": "ES",
        "x-operator": "Gutro",
        "x-path": "/auth",
        "x-request-id": req_id, # Dynamic request ID
        "Referer": "https://www.leovegas.es/auth?intent=login"
    }

    # El 'blackbox' es un valor dinámico. Por ahora, usaremos un placeholder.
    # En un entorno real, esto requeriría una lógica más compleja para generarlo.
    login_payload = {
        "operationName": "useLoginWithUsernamePasswordMutation",
        "variables": {
            "input": {
                "username": email,
                "password": password,
                "blackbox": """W;6.11.1;9JNGRxwd/FSO5J3K0Iy+aA==;GC5IOwf5Do7Y99KBdFNvI+CXr7TIkVXWdEYuJwhwUaC9FuusJJBGydMBxJfXFJl5e5mTHffzGXmB1+yKHsD5KEEURo8Hu5QDi13+wA35bhoYd7ufR/SpW+rtQBJWSyMsGOYMBRgfLihC38cOev1zz9tLEo4UTudzBtD5SbMiJ4/2lSRQBoJKn95QVgtASZ5Ug/8m1hj24Xdig230VWq7sUpQrDR7HxReBLnXZGFTrJH8May3IIXuK/+emdgCZ5vdW1gjN+fXmf31rYetmv1A7+ZD4aeQPvHjtbykyu78gLJB/5M5FYNauHsRulNbxsaphv2Am+emVi5LaKMZ0Q25nSeR3+b5T1axKVgVKjdUZa6BEIs7ro81LXV8ONcvGZt3srFaA4rMjdTGQqeN7pSFRNyRDPpndJHmHTZCJM1bKtI6toB7qyV5ejbkrda25xvZh6q6xlvezUu9GCGQ3RdFSfk3XR+L7FVPq8zK6aIWSZnwQTCL7YMANCmsrT72iWrcLiQulyny9V36PV3NzLwMPihtEgG6M9iUBfTAu0l0pOKXtyNzTRlWKsegQ+5UevBl6jtF1Gg0qcQPMXI5osUs+gtoUQNgDsjyRoP9lbVbiHCh10rtnbPXOfHAm5v6n6dqaJ6lv+DXxrTGURhhDErG""",
                "operatorUid": "Gutro" # Add operatorUid field
            }
        },
        "extensions": {"persistedQuery": {"version": 1, "sha256Hash": "faca865d80f0a00e9cd06bba9074dafa"}}
    }

    # Inicializamos una sesión de requests para que maneje las cookies automáticamente
    login_session = requests.Session()
    login_session.headers.update(login_headers)

    print(f"[*] Intentando login para {email}...")
    response = login_session.post(login_url, json=login_payload)
    print(f"[+] Login Status: {response.status_code}")

    if response.status_code == 200:
        # El AuthTokenV2 debería estar en las cookies de la respuesta
        auth_token = login_session.cookies.get('AuthTokenV2')
        if auth_token:
            print(f"[+] AuthTokenV2 obtenido exitosamente: {auth_token}")
            return auth_token, login_session.cookies
        else:
            print("[-] Error: AuthTokenV2 no encontrado en las cookies de la respuesta.")
            print(response.json()) # Mostrar el cuerpo de la respuesta para depuración
    else:
        print(f"[-] Error en el login: {response.text}")
    return None, None

# Ejecutar el login
new_auth_token, new_cookies = perform_login(LEOVEGAS_EMAIL, LEOVEGAS_PASSWORD)

if new_auth_token:
    # Actualizar la cookie AuthTokenV2 en el diccionario cookies global
    # Esto se hará en la celda f8993e5f, pero aquí guardamos el token para su uso.
    # print(f"Por favor, actualiza manualmente la celda f8993e5f con el siguiente token: {new_auth_token}")
    # Guardamos el token y las cookies en variables globales para la siguiente celda
    global DYNAMIC_AUTH_TOKEN
    global DYNAMIC_COOKIES
    DYNAMIC_AUTH_TOKEN = new_auth_token
    DYNAMIC_COOKIES = new_cookies
    print("[+] Login dinámico completado. El token y las cookies están disponibles para la configuración de la sesión.")
else:
    print("[-] Fallo en el login dinámico. No se pudo obtener el AuthTokenV2.")

[*] Intentando login para whilmisweb@gmail.com...
[+] Login Status: 200
[+] AuthTokenV2 obtenido exitosamente: 43ee440e5ba36164
[+] Login dinámico completado. El token y las cookies están disponibles para la configuración de la sesión.


In [ ]:
import requests
import json
import time
import uuid
# from google.colab import userdata # Importar userdata (comentado ya que no se usa directamente)

# --- CONFIGURACIÓN CON SALDO MANUAL ---
CONFIG = {
    'url_graphql': 'https://www.leovegas.es/api/graphql',
    'url_telemetry': 'https://browser-intake-datadoghq.eu/api/v2/rum',
    'application_id': 'cb64c2d7-e2ef-4fb2-be0a-94b813dc7e29',
    'dd_api_key': 'pubfa4404dc4ea7564c151e8aa3193797af',
    'manual_balance_str': '230,23',  # <--- ESCRIBE TU SALDO EN FORMATO EU (ej. 203,23)
    'time_limit_in_minutes': 60, # Tiempo de juego en minutos
    'reminder_time_period_in_minutes': 15, # Intervalo de recordatorio en minutos
    # Configuración para WebSocket
    'websocket_base_url': 'wss://livecasinoes.leovegas.com/public/horizon',
    'websocket_gametype': 'rng-blackjack',
    'websocket_tableid': 'rng-bj-leovegas0',
    'websocket_instance_prefix': '0bmqyz',
    'websocket_client_version': '6.20260713.73621.63295-319e59d2b8-r2',
    # Nuevas configuraciones del sistema de Blackjack
    'url_blackjack': 'https://www.leovegas.es/juegos/leovegas-blackjack',
    'url_game_log': 'https://livecasinoes.leovegas.com/log',
    'user_id': '400425393' # User ID capturado de los logs del usuario
}

# Conversión interna del saldo
try:
    CONFIG['manual_balance'] = float(CONFIG['manual_balance_str'].replace(',', '.'))
except:
    CONFIG['manual_balance'] = 0.0

# Cookies capturadas (se sobrescribirán con las dinámicas si están disponibles)
cookies = {
    'firstVisitAt': '1778008742051',
    '__auid': '80b71a99-17ff-4ba4-91b9-0e179886a77a',
    'preferredLanguage': 'ES',
    'AuthTokenV2': None, # Se actualizará dinámicamente con el token de login
    'incap_ses_1393_2121613': 'Qc2tQNdxKy0erIL7w+5UEwr8T2oAAAAABd3A1vvPuWIOkpsW/iLExg=='
}

headers = {
    'accept': '*/*',
    'content-type': 'application/json',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36',
    'x-domain': 'www.leovegas.es',
    'x-language': 'es-es',
    'x-market': 'ES',
    'x-operator': 'Gutro',
    'x-datadog-origin': 'rum',
    'traceparent': '00-0000000000000000e7996f541392230a-46e7c99a30f49d76-01'
}

session = requests.Session()
session.headers.update(headers)

# Si DYNAMIC_COOKIES y DYNAMIC_AUTH_TOKEN están definidos (del login dinámico)
if 'DYNAMIC_COOKIES' in globals() and DYNAMIC_COOKIES:
    session.cookies.update(DYNAMIC_COOKIES)
    print(f"[+] Cookies de sesión actualizadas con las obtenidas dinámicamente. AuthTokenV2: {session.cookies.get('AuthTokenV2')}")
else:
    # Fallback a las cookies estáticas si el login dinámico no se ejecutó o falló
    session.cookies.update(cookies)
    print("[-] Advertencia: No se encontraron cookies dinámicas. Usando cookies estáticas (puede requerir AuthTokenV2 manual).")


print(f"[+] Sesión configurada. Saldo manual: {CONFIG['manual_balance_str']}€ ({CONFIG['manual_balance']} float)")

[+] Cookies de sesión actualizadas con las obtenidas dinámicamente. AuthTokenV2: 43ee440e5ba36164
[+] Sesión configurada. Saldo manual: 230,23€ (230.23 float)


#### **IMPORTANTE:**

Ahora que hemos obtenido el `AuthTokenV2` de forma dinámica, necesitamos actualizar la configuración de la sesión con este nuevo token.

La celda `f8993e5f` ha sido modificada para que tome el `AuthTokenV2` de la variable global `DYNAMIC_AUTH_TOKEN` (establecida por la celda de login anterior).

**Por favor, ejecuta la celda `f8993e5f` nuevamente.** Luego, procede a ejecutar la celda `99fee72b` para refrescar el contexto de la sesión con el token válido.

In [ ]:
import requests
import json

# --- 4. Simulando Peticiones de Contexto Post-Login ---
# Usaremos el objeto 'session' que ya tiene las cookies del login previo.

def run_context_queries(session, auth_token):
    graphql_url = "https://www.leovegas.es/api/graphql"

    # Actualizamos cabeceras específicas detectadas en tus logs
    extra_headers = {
        'x-domain': 'www.leovegas.es',
        'x-language': 'es-es',
        'x-market': 'ES',
        'x-operator': 'Gutro',
        'content-type': 'application/json'
    }
    session.headers.update(extra_headers)

    # 1. Simulación de ApplicationQuery
    print("[*] Ejecutando ApplicationQuery...")
    app_payload = {
        "operationName": "ApplicationQuery",
        "variables": {
            "authenticated": True,
            "DGOJ": True, # Mercado regulado español
            "AAMS": False, "BRZ": False, "CAON": False, "CEG": False, "DGA": False, "IOM": False, "KSA": False, "MGA": False, "SGA": False, "SH": False, "UKGC": False
        },
        "extensions": {"persistedQuery": {"version": 1, "sha256Hash": "b41a6ebc7b0fc9366ba7411af2817c26"}}
    }

    resp_app = session.post(graphql_url, json=app_payload)
    print(f"[+] ApplicationQuery Status: {resp_app.status_code}")

    # 2. Simulación de PrerequisitesContextProviderQuery
    print("\n[*] Ejecutando PrerequisitesContextProviderQuery...")
    prereq_payload = {
        "operationName": "PrerequisitesContextProviderQuery",
        "variables": {"authenticated": True},
        "extensions": {"persistedQuery": {"version": 1, "sha256Hash": "2a5da453eba1d071bb0cd2d64293a1d7"}}
    }

    resp_pre = session.post(graphql_url, json=prereq_payload)
    print(f"[+] Prerequisites Status: {resp_pre.status_code}")

    # Mostramos un fragmento de la respuesta para verificar el estado
    if resp_pre.status_code == 200:
        print("[+] Datos de contexto recibidos correctamente.")
        # print(json.dumps(resp_pre.json(), indent=2)) # Opcional: ver todo el JSON

# Ejecutar si el login anterior fue exitoso
try:
    # Extraemos el token directamente de las cookies de la sesión
    current_token = session.cookies.get('AuthTokenV2')
    if current_token:
        run_context_queries(session, current_token)
        print("\n[!] Contexto de sesión actualizado. Listo para buscar datos de balance o entrar a mesas.")
    else:
        print("[-] Error: No se encontró 'AuthTokenV2' en las cookies de la sesión. Asegúrate de que la celda de login (f8993e5f) se ejecutó correctamente.")
except Exception as e:
    print(f"[-] Error durante la actualización de contexto: {e}")

[*] Ejecutando ApplicationQuery...
[+] ApplicationQuery Status: 200

[*] Ejecutando PrerequisitesContextProviderQuery...
[+] Prerequisites Status: 200
[+] Datos de contexto recibidos correctamente.

[!] Contexto de sesión actualizado. Listo para buscar datos de balance o entrar a mesas.


### Etapa 2: Game Flow y Sincronización de Saldo (Wallet)
Esta celda busca sincronizar la sesión con los servicios de billetera para detectar el saldo real disponible.

In [ ]:
print('[*] Etapa 2: Sincronización de Saldo (Modo Manual)')

# Sincronizamos el valor directamente del diccionario CONFIG global
saldo_visual = CONFIG.get('manual_balance_str', '0,00')
saldo_float = CONFIG.get('manual_balance', 0.0)

print('\n' + '='*40)
print('   ESTADO DE CUENTA (CONFIGURACIÓN MANUAL)')
print('='*40)
print(f'[+] Billetera Cash   | {saldo_visual:>8} EUR')
print('='*40)
print(f'[!] El bot operará con el valor numérico: {saldo_float}')

[*] Etapa 2: Sincronización de Saldo (Modo Manual)

   ESTADO DE CUENTA (CONFIGURACIÓN MANUAL)
[+] Billetera Cash   |   230,23 EUR
[!] El bot operará con el valor numérico: 230.23


In [ ]:
import requests
import json
import time
import uuid
import random

# --- CONFIGURATION (already updated in f8993e5f, but for local reference if needed) ---
# The CONFIG dictionary is now global from f8993e5f

# --- ASSETS (from user provided code) ---
ASSETS = {
    'win_icon': "data:image/svg+xml,%3csvg xmlns='http://www.w3.org/2000/svg' viewBox='-1 -3 30 30'%3e%3cpath fill='%23bb9c15' d='M21.05 20.87c.864 0 1.565.7 1.565 1.565S21.915 24 21.05 24H6.95c-.864 0-1.565-.7-1.565-1.565s.7-1.565 1.565-1.565h14.1z'/%3e%3c/svg%3e",
    'loss_icon': "data:image/svg+xml,%3csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 30 30'%3e%3cpath fill='%23bbb' d='M43 9h-6l-7 8-7-8h-6l9 11-9 11h6l7-8 7 8h6l-9-11z'/%3e%3c/svg%3e"
}

# --- Helper Functions (from user provided code) ---
def get_basic_strategy_action(player_sum, dealer_card):
    if player_sum <= 8: return 'HIT', "Mano débil (<=8), pedir."
    if player_sum == 9: return ('DOUBLE', "Doblamos con 9") if 3 <= dealer_card <= 6 else ('HIT', "Pedir con 9")
    if player_sum == 10: return ('DOUBLE', "Doblamos con 10") if dealer_card <= 9 else ('HIT', "Pedir con 10")
    if player_sum == 11: return 'DOUBLE', "11: Doblar siempre."
    if player_sum == 12: return ('STAND', "12 vs débil") if 4 <= dealer_card <= 6 else ('HIT', "12 vs fuerte")
    if 13 <= player_sum <= 16: return ('STAND', "13-16 vs débil") if dealer_card <= 6 else ('HIT', "Riesgo controlado")
    return 'STAND', f"{player_sum}: Plantarse."

def send_user_heartbeat(session_obj):
    ts = int(time.time() * 1000)
    params = {'ddsource': 'browser', 'dd-api-key': CONFIG['dd_api_key'], 'batch_time': ts}
    payload = {"type": "view", "usr": {"id": CONFIG['user_id']}, "view": {"url": CONFIG['url_blackjack']}, "application": {"id": CONFIG['application_id']}}
    try: session_obj.post(CONFIG['url_telemetry'], params=params, json=payload, timeout=3)
    except: pass

print('[*] Ejecutando Etapa 3: Navegación y Preparación...')
try:
    # Safety check: Ensure 'session' object is available. If not, re-initialize.
    if 'session' not in globals():
        print("[-] WARNING: 'session' object not found in global scope. "
              "Creating a new basic requests.Session(). "
              "For full functionality, please ensure cell f8993e5f was executed to configure it with dynamic AuthTokenV2 and headers.")
        session = requests.Session()
        # Attempt to set basic headers if 'headers' are available, otherwise a bare session.
        if 'headers' in globals():
            session.headers.update(headers)
        # If 'DYNAMIC_COOKIES' exist, try to apply them
        if 'DYNAMIC_COOKIES' in globals():
            session.cookies.update(DYNAMIC_COOKIES)

    # Explicitly set EVOSESSIONID from the user's provided context
    session.cookies.set('EVOSESSIONID', 'tx4zawu5evztwsryt5rpbayipe4lz4if625f75461e456c2b81233707c6364a7ec0308ba2824f7a84')
    print(f'[+] EVOSESSIONID configurado manualmente: {session.cookies.get("EVOSESSIONID")}')

    # 1. Navegación al lobby de casino
    game_url = "https://www.leovegas.es/casino-en-vivo/blackjack"
    print(f'[+] Accediendo a la cabecera del juego...')
    # Cambiado allow_redirects a True para permitir que se establezca el EVOSESSIONID
    session.get(game_url, allow_redirects=True)

    # Check EVOSESSIONID after game_url GET request
    evosessionid_after_get = session.cookies.get('EVOSESSIONID', 'NOT_FOUND')
    print(f'[+] EVOSESSIONID después de acceder a game_url: {evosessionid_after_get}')

    # 2. Formulario de Entrada (DGOJ) - ENVÍO DE LÍMITES DE JUEGO
    print('[*] Enviando formulario de cumplimiento DGOJ (saldo, tiempo de juego, notificación)...')
    dgoj_payload = {
        "operationName": "groupApiGQLGameLaunchMutation",
        "variables": {
            "input": {
                "gameId": "10041", # Asumiendo este es el gameId para Blackjack de Leovegas
                "session": {
                    "lossLimit": CONFIG['manual_balance'],
                    "timeLimitInMinutes": CONFIG['time_limit_in_minutes'],
                    "reminderTimePeriodInMinutes": CONFIG['reminder_time_period_in_minutes']
                }
            }
        },
        "extensions": {"persistedQuery": {"version": 1, "sha256Hash": "43bcffa336c57ee7f4a321115805ac6f"}}
    }

    # Headers específicos para esta petición, si los hubiera, se añadirían aquí.
    # Por ahora, usamos los de la sesión.
    dgoj_response = session.post(CONFIG['url_graphql'], json=dgoj_payload)
    print(f"[+] Formulario DGOJ Status: {dgoj_response.status_code}")
    if dgoj_response.status_code == 200:
        print("[+] Formulario DGOJ enviado y aceptado.")
        print("[+] Respuesta completa del formulario DGOJ:")
        print(json.dumps(dgoj_response.json(), indent=2))
        # Nuevo: Verificar EVOSESSIONID después de enviar el formulario DGOJ
        evosessionid_after_dgoj = session.cookies.get('EVOSESSIONID', 'NOT_FOUND')
        print(f'[+] EVOSESSIONID después de enviar formulario DGOJ: {evosessionid_after_dgoj}')
    else:
        print(f"[-] Error al enviar formulario DGOJ: {dgoj_response.text}")

    # 3. Telemetría (Crucial para mantener la sesión 'viva')
    print('[*] Enviando pulso de actividad a Datadog...')
    telemetry_payload = {
        "events": [{
            "date": int(time.time() * 1000),
            "type": "telemetry",
            "view": {"id": str(uuid.uuid4())},
            "session": {"id": str(uuid.uuid4())},
            "application": {"id": CONFIG['application_id']},
            "telemetry": {"type": "log", "status": "ok", "message": "User engaged with game table"}
        }]
    }
    session.post(CONFIG['url_telemetry'], json=telemetry_payload, params={'dd-api-key': CONFIG['dd_api_key']})

    print('\n[READY] Sesión preparada. Esperando instrucciones para la lógica de apuestas.')

    # --- Blackjack System (from user provided code, adapted to use existing session) ---
    def run_blackjack_system(session_obj):
        s = session_obj  # Use the passed session object

        # HÍBRIDO MARTINGALE + D'ALEMBERT
        base_bet = 1
        current_bet = base_bet
        consecutive_losses = 0

        print(f"[*] Iniciando sistema híbrido. Apuesta base: {base_bet}€")

        for i in range(1, 11):
            send_user_heartbeat(s)  # Pass the session object
            print(f"\n--- [MANO {i}/10] | APUESTA: {current_bet}€ ---")

            p_hand = [random.randint(2, 11), random.randint(2, 10)]
            d_up = random.randint(2, 11)
            p_sum = sum(p_hand)

            # Simulación de juego con estrategia básica
            while p_sum < 21:
                action, reason = get_basic_strategy_action(p_sum, d_up)
                if action in ['STAND', 'DOUBLE']: break
                p_sum += random.randint(2, 10)

            # Determinar resultado (Simulado para demo)
            outcome = random.choice(['WIN', 'LOSS'])

            if outcome == 'LOSS':
                consecutive_losses += 1
                print(f"[RESULTADO] DERROTA. (Racha: {consecutive_losses})")

                if consecutive_losses >= 4:
                    print("[!] LÍMITE DE 4 PÉRDIDAS ALCANZADO. Reiniciando a base (1€).")
                    current_bet = base_bet
                    consecutive_losses = 0
                else:
                    print(f"[MARTINGALE] Doblando apuesta: {current_bet}€ -> {current_bet * 2}€")
                    current_bet *= 2
            else:
                print("[RESULTADO] VICTORIA.")
                consecutive_losses = 0
                # D'Alembert: Reducción gradual tras ganar
                old_bet = current_bet
                current_bet = max(base_bet, current_bet - 1)
                print(f"[D'ALEMBERT] Reduciendo apuesta: {old_bet}€ -> {current_bet}€")

            # Enviar log de apuesta al servidor
            # Dynamically get EVOSESSIONID from the session cookies
            evo_session_id = s.cookies.get('EVOSESSIONID', 'NOT_FOUND_FOR_LOG')
            if evo_session_id == 'NOT_FOUND_FOR_LOG':
                print("[-] EVOSESSIONID no encontrado en las cookies de la sesión. El log de apuesta no se enviará.")
                bet_log = {"log": {"type": "USER_ACTION", "value": {"action": "PLACE_BET", "amount": current_bet, "tableId": "rng-bj-leovegas0", "EVOSESSIONID_MISSING": True}}}
            else:
                bet_log = {"log": {"type": "USER_ACTION", "value": {"action": "PLACE_BET", "amount": current_bet, "tableId": "rng-bj-leovegas0"}}}

            try: s.post(CONFIG['url_game_log'], params={'EVOSESSIONID': evo_session_id}, json=bet_log, timeout=3)
            except Exception as e: print(f"[-] Error al enviar log de apuesta: {e}")

            time.sleep(random.randint(5, 8))

        print("\n[*] Sincronizando balance final...")
        save_payload = {"operationName": "SessionContextGameSessionLimitQuery", "variables": {}, "extensions": {"persistedQuery": {"version": 1, "sha256Hash": "7c88206bdc77aefa880630c1bf68675e"}}}
        s.post(CONFIG['url_graphql'], json=save_payload)
        print("[!] PROCESO COMPLETADO.")

    # Call the blackjack system
    print("\n[*] Iniciando sistema de juego de Blackjack...")
    run_blackjack_system(session)

except Exception as e:
    print(f'[-] Error en Etapa 3: {e}')

[*] Ejecutando Etapa 3: Navegación y Preparación...
[+] EVOSESSIONID configurado manualmente: tx4zawu5evztwsryt5rpbayipe4lz4if625f75461e456c2b81233707c6364a7ec0308ba2824f7a84
[+] Accediendo a la cabecera del juego...
[+] EVOSESSIONID después de acceder a game_url: tx4zawu5evztwsryt5rpbayipe4lz4if625f75461e456c2b81233707c6364a7ec0308ba2824f7a84
[*] Enviando formulario de cumplimiento DGOJ (saldo, tiempo de juego, notificación)...
[+] Formulario DGOJ Status: 200
[+] Formulario DGOJ enviado y aceptado.
[+] Respuesta completa del formulario DGOJ:
{
  "errors": [
    {
      "message": "Invalid status code: 409",
      "path": [
        "groupApiGQL_gameLaunch"
      ],
      "extensions": {
        "code": "INTERNAL_SERVER_ERROR",
        "cause": {
          "message": "(instance id = 3FCAB8759DD178E0, category code = confliction, error code = null, source = game@game-688758dc-4f45b) Game 10041 is not configured for fun"
        },
        "http": {
          "status": 500
        },
    